# DistilBERT — Phát hiện tin giả COVID-19
Lương Hoàng Bảo Quân — 24DH111520 · Nhóm 5

Chạy từng ô. Bật Internet và GPU trước. Chưa có output thực nghiệm trong notebook này. Mỗi phiên nên hoàn thành và lưu một seed trước khi chạy seed tiếp theo.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/Doris1808/covid19-fake-news-nhom5.git"
PROJECT = Path("/kaggle/working/covid19-fake-news-nlp")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
# Chỉ dùng một GPU để giữ effective batch đúng với config.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Thư mục làm việc:", Path.cwd())


## Cài thư viện
Nếu có yêu cầu restart kernel, khởi động lại và chạy lại ô đường dẫn phía trên. Không cài lại giữa lúc đang train.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import torch, transformers
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU CUDA. Kiểm tra Accelerator trước khi train trên Kaggle.")
print("GPU:", torch.cuda.get_device_name(0))


## Tải dữ liệu và xem thống kê
Giữ nguyên ba split. Bản CSV là mirror cộng đồng; đối chiếu với nguồn nhóm thống nhất. EDA chỉ mô tả Test, không dùng để chọn cấu hình theo kết quả Test.

In [ ]:
subprocess.run([sys.executable, "-m", "src.download_data"], check=True)
subprocess.run([sys.executable, "-m", "src.eda"], check=True)
import pandas as pd
pd.read_csv("outputs/eda/dataset_stats.csv")


In [ ]:
pd.read_csv("outputs/eda/examples_train.csv")


## Chạy thử ít mẫu
Mục đích kiểm tra train → lưu → nạp checkpoint. Không tính lượt này vào ba seed chính thức. Nếu đã chạy xong thì bỏ qua ô này hoặc chọn output khác.

In [ ]:
subprocess.run([sys.executable, "-m", "src.finetune", "--seed", "42", "--smoke"], check=True)


In [ ]:
from src.predict import load_model, predict
tokenizer, model = load_model("outputs/smoke/seed_42/model")
print(predict("Example text used only to check model loading.", tokenizer, model))
del tokenizer, model


## Train toàn bộ Train — một seed mỗi lần
Chạy 42 trước; lưu output rồi thay bằng 123 và 2024. Không sửa cấu hình giữa các seed dùng để tính trung bình. Model mới bắt đầu từ checkpoint gốc ở mỗi seed.

In [ ]:
SEED = 42
subprocess.run([sys.executable, "-m", "src.finetune", "--seed", str(SEED)], check=True)


In [ ]:
import json
run_dir = Path("outputs/distilbert") / f"seed_{SEED}"
metrics = json.loads((run_dir / "metrics.json").read_text())
print(json.dumps(metrics, indent=2, ensure_ascii=False))


## Tổng hợp và lưu kết quả
ZIP gồm model và checkpoint nên khá lớn. Tải về trước khi kết thúc phiên, đồng thời lưu phiên bản notebook/output trên Kaggle. Chạy ô này sau mỗi seed, kể cả chưa đủ ba seed.

In [ ]:
subprocess.run([sys.executable, "-m", "src.summarize"], check=True)
import shutil
archive = shutil.make_archive("/kaggle/working/distilbert_outputs", "zip", root_dir=PROJECT, base_dir="outputs")
from IPython.display import FileLink, display
display(FileLink(archive))


## Đánh giá Test
Chỉ thực hiện sau khi nhóm chốt cấu hình. Làm theo `docs/KAGGLE_VA_MAC.md`; notebook train này không tự chạy Test.

## Minh chứng của Quân — Nhóm 5
GPU, seed, thời gian, cấu hình và metric được lưu tự động trong `environment.json`, `run_config.json` và `metrics.json` của từng lượt chạy. Xem bảng `outputs/distilbert/summary.md` sau ô tổng hợp.
Lỗi phát sinh trong bước train được ghi ở `failure.json`; nhóm bổ sung nhận xét nguyên nhân và cách xử lý dựa trên lần chạy thực tế.
